# Tarefa 3 - Floresta aleatoria inicial

**Equipe 18** - Bruno Alves Ribeiro de Souza, Igor da Silva Alves Correa, Pedro Henrique Alexandre da Silva, Samuel de Oliveira Santos, Victoria Agatha Rodrigues Fagundes, Wendel Henrique da Silva Rocha
**Scrum Master da tarefa:** ____________

> A Tarefa 2 fez a ficha e o rotulo. Este notebook **nao refaz** nenhum dos dois: le o treino e a validacao ja cortados.
>
> **O arquivo de teste nao e aberto aqui.** Ele fica fechado ate a Tarefa 5. A escolha de hiperparametros usa **so a validacao**.

| | Artefato | Origem / destino |
|---|---|---|
| **Entra** | Ficha de baseline, dataset rotulado e corte temporal (treino.csv, validacao.csv) | Tarefa 2 - os mesmos |
| **Sai** | Floresta aleatoria treinada so no treino, com hiperparametros registrados | Base para comparacao com a Tarefa 4 |
| **Sai** | Importancia das features, exemplo de arvore componente e matriz 3x3 | Tarefa 4 |
| **Sai** | Florestas de contraste (com rota ou RTT absoluto) e analise do uso desses atalhos | Prova de que o modelo oficial nao depende do atalho |

## 0. Bibliotecas, caminhos e parametros

Bibliotecas externas: **pandas**, **numpy** e **scikit-learn (>= 1.4)**. A versao 1.4 e necessaria porque o Random Forest do scikit-learn passou a aceitar valores vazios (NaN) nas features; assim o RTT ausente continua vazio, sem ser trocado por 0.

In [22]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.tree import export_text

print("pandas", pd.__version__, "| numpy", np.__version__, "| scikit-learn", sklearn.__version__)
_v = tuple(int(x) for x in sklearn.__version__.split(".")[:2])
assert _v >= (1, 4), "Atualize o scikit-learn (>= 1.4): o Random Forest precisa aceitar NaN nas features."

pandas 3.0.5 | numpy 2.5.3 | scikit-learn 1.9.1


In [23]:
# A raiz do repositório é localizada sozinha: o notebook funciona de qualquer subpasta (ex.: notebook/arvore/).
CURRENT_DIR = Path.cwd().resolve()
ROOT = next((c for c in (CURRENT_DIR, *CURRENT_DIR.parents) if (c / "data_ripe_atlas" / "processed").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Não encontrei data_ripe_atlas/processed a partir da pasta atual.")

PASTA_SPLITS = ROOT / "data_ripe_atlas" / "processed"
CAMINHO_TREINO    = PASTA_SPLITS / "treino.csv"
CAMINHO_VALIDACAO = PASTA_SPLITS / "validacao.csv"
CAMINHO_BASELINE  = PASTA_SPLITS / "baseline_por_fluxo.csv"      # o baseline commitado na Tarefa 2
PASTA_MODELOS = ROOT / "data_ripe_atlas" / "models"              # saída: floresta, importancia e parametros
# O arquivo de teste NÃO é lido neste notebook (fica fechado até a Tarefa 5).

CLASSES = ["OK", "RISCO", "FALHA"]          # ordem de leitura da matriz 3x3
ALVO = "classe"
FEATURES = ["z_robusto", "aumento_pct", "jitter_relativo", "perda_pct",
            "timeout_atual", "n5_timeout", "n5_aumento80", "n5_risco"]
NAO_PODEM_ENTRAR = ["pais_origem", "pais_destino", "rota", "dst_addr", "prb_id", "fluxo_id",
                    "avg", "rtt_min_ms", "rtt_max_ms", "jitter_ms", "mediana", "MAD", "jitter_tipico"]

CRITERIO = "gini"                            # criterio de impureza das arvores componentes
SEMENTE = 42                                 # random_state para reproducibilidade da floresta
N_ESTIMATORS = 100
GRADE_PROFUNDIDADE = [6, 10, 14]
GRADE_MIN_FOLHA = [1, 10, 50]
TOLERANCIA_F1 = 0.005                        # escolhe a menor floresta dentro desta tolerancia do melhor F1 macro

def milhar(n): return f"{int(n):,}".replace(",", ".")
def br(x, casas=2): return f"{x:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")

## 1. Colunas que a floresta oficial pode usar

Entram: z_robusto, aumento_pct, jitter_relativo, perda_pct, timeout_atual, n5_timeout, n5_aumento80 e n5_risco.
Nao entram: pais, IP, rota, fluxo_id, nome do destino ou RTT em milissegundos no lugar das metricas relativas.
O alvo e a coluna classe; ela **nao** e feature.

- [ ] Treino, validacao e teste sao os da Tarefa 2
- [ ] A ficha de baseline nao foi recalculada com o Periodo B
- [ ] Lista de colunas do treino colada no diario
- [ ] Conferido que a classe e o alvo, nao uma feature

In [24]:
# Recalcula os caminhos para não reutilizar valores antigos da sessão do kernel.
CURRENT_DIR = Path.cwd().resolve()
ARQUIVOS_ENTRADA = ("treino.csv", "validacao.csv", "baseline_por_fluxo.csv")
ROOT = next((c for c in (CURRENT_DIR, *CURRENT_DIR.parents)
             if all((c / "data_ripe_atlas" / "processed" / nome).is_file() for nome in ARQUIVOS_ENTRADA)), None)
if ROOT is None:
    raise FileNotFoundError("Não encontrei treino.csv, validacao.csv e baseline_por_fluxo.csv em data_ripe_atlas/processed a partir da pasta atual.")
PASTA_SPLITS = ROOT / "data_ripe_atlas" / "processed"
CAMINHO_TREINO = PASTA_SPLITS / "treino.csv"
CAMINHO_VALIDACAO = PASTA_SPLITS / "validacao.csv"
CAMINHO_BASELINE = PASTA_SPLITS / "baseline_por_fluxo.csv"

tr = pd.read_csv(CAMINHO_TREINO)
va = pd.read_csv(CAMINHO_VALIDACAO)
base = pd.read_csv(CAMINHO_BASELINE)

# 1) São os blocos da Tarefa 2?
print("Treino    N:", milhar(len(tr)), "| conjunto:", sorted(tr["conjunto"].unique()), "| periodo:", sorted(tr["periodo"].unique()))
print("Validação N:", milhar(len(va)), "| conjunto:", sorted(va["conjunto"].unique()), "| periodo:", sorted(va["periodo"].unique()))
assert set(tr["conjunto"]) == {"treino"} and set(va["conjunto"]) == {"validacao"}
assert set(tr["periodo"]) == {"B"} and set(va["periodo"]) == {"B"}, "Há registro do Período A"
lim_tr = tr.groupby("fluxo_id")["timestamp"].max()
lim_va = va.groupby("fluxo_id")["timestamp"].min()
assert (lim_tr < lim_va.reindex(lim_tr.index)).all(), "Treino e validação se misturam em algum fluxo"
print("Em cada fluxo, o treino é anterior à validação: OK")
print("Esperado na Tarefa 2: treino 95.406 e validação 38.138 | obtido:", milhar(len(tr)), "e", milhar(len(va)))

# 2) A ficha é a congelada? (a ficha repetida em cada linha precisa ser a do baseline commitado)
ficha_nas_linhas = pd.concat([tr, va]).groupby("fluxo_id")[["mediana", "MAD", "jitter_tipico"]].agg(["nunique"])
assert (ficha_nas_linhas.to_numpy() == 1).all(), "A ficha muda dentro de um fluxo"
cmp = (pd.concat([tr, va]).drop_duplicates("fluxo_id").set_index("fluxo_id")[["mediana", "MAD", "jitter_tipico"]]
       .join(base.set_index("fluxo_id")[["mediana", "MAD", "jitter_tipico"]], rsuffix="_baseline", how="outer"))
assert cmp.notna().all().all(), "Fluxos diferentes entre o dataset e o baseline"
for c in ("mediana", "MAD", "jitter_tipico"):
    assert np.allclose(cmp[c], cmp[c + "_baseline"], rtol=0, atol=1e-6), f"{c} difere do baseline"
print(f"Ficha das linhas == baseline_por_fluxo.csv ({len(base)} fluxos): OK")

# 3) Colunas
assert ALVO not in FEATURES, "O alvo não pode ser feature"
assert not set(FEATURES) & set(NAO_PODEM_ENTRAR), "Há coluna proibida entre as features"
print()
print("Colunas usadas pela floresta oficial:", ", ".join(FEATURES))
print("Alvo:", ALVO)
print("Colunas do treino que ficam de fora (auditoria ou proibidas):", ", ".join(c for c in tr.columns if c not in FEATURES + [ALVO]))
print()
print("Classes no treino:", tr[ALVO].value_counts().reindex(CLASSES).to_dict())
print("Classes na validação:", va[ALVO].value_counts().reindex(CLASSES).to_dict())
print("Valores vazios nas features (treino):", {c: int(tr[c].isna().sum()) for c in FEATURES if tr[c].isna().any()})

X_tr, y_tr = tr[FEATURES], tr[ALVO]
X_va, y_va = va[FEATURES], va[ALVO]

Treino    N: 95.406 | conjunto: ['treino'] | periodo: ['B']
Validação N: 38.138 | conjunto: ['validacao'] | periodo: ['B']
Em cada fluxo, o treino é anterior à validação: OK
Esperado na Tarefa 2: treino 95.406 e validação 38.138 | obtido: 95.406 e 38.138
Ficha das linhas == baseline_por_fluxo.csv (76 fluxos): OK

Colunas usadas pela floresta oficial: z_robusto, aumento_pct, jitter_relativo, perda_pct, timeout_atual, n5_timeout, n5_aumento80, n5_risco
Alvo: classe
Colunas do treino que ficam de fora (auditoria ou proibidas): fluxo_id, prb_id, dst_addr, measurement_id, timestamp, datetime_utc, avg, rtt_min_ms, rtt_max_ms, sent, rcvd, jitter_ms, pais_origem, pais_destino, rota, periodo, mediana, MAD, jitter_tipico, crit_risco_ind, conjunto

Classes no treino: {'OK': 72857, 'RISCO': 6582, 'FALHA': 15967}
Classes na validação: {'OK': 27180, 'RISCO': 3388, 'FALHA': 7570}
Valores vazios nas features (treino): {'z_robusto': 197, 'aumento_pct': 197, 'jitter_relativo': 261}


## 2. A floresta aleatoria oficial

Random Forest de classificacao com criterio **Gini**, n_estimators fixo em 100 e max_depth/min_samples_leaf escolhidos olhando **so a validacao**, pelo **F1 macro** (a acuracia sozinha nao decide: com maioria OK ela fica alta mesmo errando FALHA). Entre as configuracoes ate TOLERANCIA_F1 abaixo do melhor F1 macro, escolhemos a menor floresta pelo total de folhas de suas arvores. O fit e feito **so no treino**.

As regras de uma arvore componente sao apenas exemplos: as previsoes oficiais resultam da votacao de todas as arvores da floresta.

- [ ] Floresta aleatoria (100 arvores, Gini)
- [ ] max_depth e min_samples_leaf escolhidos olhando so a validacao
- [ ] Semente registrada
- [ ] Ajuste (fit) so no treino
- [ ] Profundidade e folhas medias das arvores componentes e importancia das features reportadas
- [ ] Exemplo de arvore componente identificado como ilustrativo

In [25]:

def nova_floresta(max_depth, min_samples_leaf):
    return RandomForestClassifier(
        n_estimators=N_ESTIMATORS,
        criterion=CRITERIO,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=SEMENTE,
        n_jobs=-1,
    )


linhas = []
for md_ in GRADE_PROFUNDIDADE:
    for msl in GRADE_MIN_FOLHA:
        modelo = nova_floresta(md_, msl).fit(X_tr, y_tr)
        pred = modelo.predict(X_va)
        linhas.append({
            "max_depth": md_,
            "min_samples_leaf": msl,
            "f1_macro_val": f1_score(y_va, pred, labels=CLASSES, average="macro", zero_division=0),
            "acuracia_val": float((pred == y_va).mean()),
            "profundidade_media": float(np.mean([arvore.get_depth() for arvore in modelo.estimators_])),
            "folhas_totais": int(sum(arvore.get_n_leaves() for arvore in modelo.estimators_)),
        })

grade = pd.DataFrame(linhas)
melhor = grade["f1_macro_val"].max()
candidatas = grade[grade["f1_macro_val"] >= melhor - TOLERANCIA_F1]
escolhida = candidatas.sort_values(
    ["folhas_totais", "max_depth", "min_samples_leaf"],
    ascending=[True, True, False],
).iloc[0]
MAX_DEPTH = int(escolhida["max_depth"])
MIN_FOLHA = int(escolhida["min_samples_leaf"])

print(f"Melhor F1 macro na validacao: {br(melhor, 4)} | tolerancia: {br(TOLERANCIA_F1, 3)} | configuracoes dentro da tolerancia: {len(candidatas)}")
print()
print("Resultados por configuracao:")
resumo_grade = grade.sort_values(["max_depth", "min_samples_leaf"])[
    ["max_depth", "min_samples_leaf", "f1_macro_val", "profundidade_media", "folhas_totais"]
].copy()
resumo_grade["f1_macro_val"] = resumo_grade["f1_macro_val"].round(4)
resumo_grade["profundidade_media"] = resumo_grade["profundidade_media"].round(2)
print(resumo_grade.to_string(index=False))
print()
print(f"ESCOLHIDA: max_depth = {MAX_DEPTH} | min_samples_leaf = {MIN_FOLHA} | F1 macro (validacao) = {br(escolhida['f1_macro_val'], 4)} | total de folhas = {int(escolhida['folhas_totais'])}")

Melhor F1 macro na validacao: 0,9994 | tolerancia: 0,005 | configuracoes dentro da tolerancia: 9

Resultados por configuracao:
 max_depth  min_samples_leaf  f1_macro_val  profundidade_media  folhas_totais
         6                 1        0.9986                6.00           2734
         6                10        0.9984                6.00           2376
         6                50        0.9957                5.96           1949
        10                 1        0.9993               10.00           9617
        10                10        0.9984               10.00           6650
        10                50        0.9958                9.73           4085
        14                 1        0.9994               13.97          15993
        14                10        0.9983               13.55           8828
        14                50        0.9959               11.85           4600

ESCOLHIDA: max_depth = 6 | min_samples_leaf = 50 | F1 macro (validacao) = 0,9957 | total de 

In [26]:
# Floresta oficial: fit so no treino
floresta = nova_floresta(MAX_DEPTH, MIN_FOLHA).fit(X_tr, y_tr)
profundidades = [arvore.get_depth() for arvore in floresta.estimators_]
folhas = [arvore.get_n_leaves() for arvore in floresta.estimators_]

print("Criterio:", CRITERIO, "| arvores:", N_ESTIMATORS, "| semente (random_state):", SEMENTE)
print("max_depth:", MAX_DEPTH, "| min_samples_leaf:", MIN_FOLHA)
print("Profundidade media das arvores componentes:", br(np.mean(profundidades)),
      "| folhas medias:", br(np.mean(folhas)))
print("F1 macro no treino (so referencia, nao decide nada):",
      br(f1_score(y_tr, floresta.predict(X_tr), labels=CLASSES, average="macro", zero_division=0), 4))
print()
print("Importancia das features na floresta:")
print(pd.Series(floresta.feature_importances_, index=FEATURES).sort_values(ascending=False).round(4).to_string())
print()
def texto_arvore_com_rotulos(modelo, arvore, nomes, max_depth):
    texto = export_text(arvore, feature_names=nomes, decimals=2, max_depth=max_depth)
    for indice, classe in enumerate(modelo.classes_):
        texto = texto.replace(f"class: {float(indice)}", f"class: {classe}")
    return texto


texto_arvore = texto_arvore_com_rotulos(floresta, floresta.estimators_[0], FEATURES, max_depth=3)
print("Exemplo ilustrativo - primeiras divisoes da arvore componente 1 (nao representa a floresta inteira):")
print(texto_arvore)

Criterio: gini | arvores: 100 | semente (random_state): 42
max_depth: 6 | min_samples_leaf: 50
Profundidade media das arvores componentes: 5,96 | folhas medias: 19,49
F1 macro no treino (so referencia, nao decide nada): 0,9957

Importancia das features na floresta:
z_robusto          0.5855
aumento_pct        0.1880
n5_risco           0.1255
jitter_relativo    0.0902
perda_pct          0.0070
n5_aumento80       0.0028
timeout_atual      0.0008
n5_timeout         0.0002

Exemplo ilustrativo - primeiras divisoes da arvore componente 1 (nao representa a floresta inteira):
|--- n5_aumento80 <= 1.50
|   |--- jitter_relativo <= 3.00
|   |   |--- aumento_pct <= 0.22
|   |   |   |--- n5_risco <= 1.50
|   |   |   |   |--- truncated branch of depth 3
|   |   |   |--- n5_risco >  1.50
|   |   |   |   |--- truncated branch of depth 2
|   |   |--- aumento_pct >  0.22
|   |   |   |--- z_robusto <= 3.50
|   |   |   |   |--- truncated branch of depth 3
|   |   |   |--- z_robusto >  3.50
|   |   |   | 

In [27]:
raizes = pd.Series([
    FEATURES[arvore.tree_.feature[0]]
    for arvore in floresta.estimators_
    if arvore.tree_.feature[0] >= 0
]).value_counts()
print("Features mais frequentes na raiz das arvores componentes:")
print(raizes.head(5).to_string())
print()
print("Exemplo de arvore componente (profundidade limitada para leitura; a previsao oficial e a votacao da floresta):")
print(texto_arvore_com_rotulos(floresta, floresta.estimators_[0], FEATURES, max_depth=3))

Features mais frequentes na raiz das arvores componentes:
aumento_pct        28
z_robusto          23
n5_risco           14
jitter_relativo    14
perda_pct           9

Exemplo de arvore componente (profundidade limitada para leitura; a previsao oficial e a votacao da floresta):
|--- n5_aumento80 <= 1.50
|   |--- jitter_relativo <= 3.00
|   |   |--- aumento_pct <= 0.22
|   |   |   |--- n5_risco <= 1.50
|   |   |   |   |--- truncated branch of depth 3
|   |   |   |--- n5_risco >  1.50
|   |   |   |   |--- truncated branch of depth 2
|   |   |--- aumento_pct >  0.22
|   |   |   |--- z_robusto <= 3.50
|   |   |   |   |--- truncated branch of depth 3
|   |   |   |--- z_robusto >  3.50
|   |   |   |   |--- class: FALHA
|   |--- jitter_relativo >  3.00
|   |   |--- n5_risco <= 1.50
|   |   |   |--- aumento_pct <= 1.30
|   |   |   |   |--- class: OK
|   |   |   |--- aumento_pct >  1.30
|   |   |   |   |--- truncated branch of depth 3
|   |   |--- n5_risco >  1.50
|   |   |   |--- z_robusto <=

## 3. Leitura do erro (validação)

O teste continua fechado. A leitura do erro usa o bloco de **validação**.

In [28]:
pred_va = floresta.predict(X_va)

mat = pd.DataFrame(confusion_matrix(y_va, pred_va, labels=CLASSES),
                   index=[f"real {c}" for c in CLASSES], columns=[f"prevista {c}" for c in CLASSES])
print("Matriz 3x3 (validação) — linhas: classe real; colunas: classe prevista pela floresta")
print(mat.to_string())
print()
p, r, f, s = precision_recall_fscore_support(y_va, pred_va, labels=CLASSES, zero_division=0)
metricas = pd.DataFrame({"precisão": p, "recall": r, "F1": f, "medições": s}, index=CLASSES)
print(metricas.round(4).to_string())
f1_macro = f1_score(y_va, pred_va, labels=CLASSES, average="macro", zero_division=0)
acuracia = float((pred_va == y_va).mean())
print()
print(f"F1 macro (validação): {br(f1_macro, 4)}")
print(f"Acurácia (só contexto): {br(acuracia * 100, 2)}% | proporção de OK na validação: {br((y_va == 'OK').mean() * 100, 2)}% "
      f"(prever sempre OK já daria essa acurácia sem acertar nenhuma FALHA)")

Matriz 3x3 (validação) — linhas: classe real; colunas: classe prevista pela floresta
            prevista OK  prevista RISCO  prevista FALHA
real OK           27180               0               0
real RISCO           17            3371               0
real FALHA           82              14            7474

       precisão  recall      F1  medições
OK       0.9964  1.0000  0.9982     27180
RISCO    0.9959  0.9950  0.9954      3388
FALHA    1.0000  0.9873  0.9936      7570

F1 macro (validação): 0,9957
Acurácia (só contexto): 99,70% | proporção de OK na validação: 71,27% (prever sempre OK já daria essa acurácia sem acertar nenhuma FALHA)


In [29]:
# Erros concretos (validação)
val = va.copy()
val["previsao"] = pred_va
# Só auditoria: qual linha da tabela de rotulagem da Tarefa 2 definiu a classe real (recalculada a partir das colunas do arquivo)
_z = val["z_robusto"].fillna(-np.inf)
val["linha_rotulo"] = np.select(
    [val["perda_pct"] >= 10, val["n5_timeout"] >= 3, _z >= 3.5, val["n5_aumento80"] >= 2,
     (val["crit_risco_ind"] == 1) & (val["n5_risco"] >= 2)], [1, 2, 3, 4, 5], default=6)
cols_erro = ["fluxo_id", "datetime_utc", "mediana", "avg", "z_robusto", "aumento_pct", "jitter_relativo",
             "perda_pct", "n5_timeout", "n5_aumento80", "crit_risco_ind", "n5_risco", "classe", "previsao", "linha_rotulo"]

longo_estavel_em_falha = val[(val["previsao"] == "FALHA") & (val["classe"] != "FALHA") & (val["mediana"] > 150)
                             & (val["z_robusto"].abs() < 2) & (val["perda_pct"] == 0)]
falha_curta_em_ok = val[(val["classe"] == "FALHA") & (val["previsao"] == "OK") & (val["mediana"] < 5)]

print("(a) Caminho longo estável (mediana > 150 ms, |z_robusto| < 2, sem perda) previsto como FALHA:", milhar(len(longo_estavel_em_falha)))
print("(b) FALHA de caminho curto (mediana < 5 ms) prevista como OK:", milhar(len(falha_curta_em_ok)))
for titulo, df_ in (("(a)", longo_estavel_em_falha), ("(b)", falha_curta_em_ok)):
    if len(df_) == 0:
        print(f"\n{titulo}: o caso não apareceu na validação.")
    else:
        print(f"\n{titulo}: primeiros exemplos (colunas mediana e avg são só auditoria):")
        print(df_.sort_values(["fluxo_id", "timestamp"]).head(2)[cols_erro].to_string(index=False))

erros = val[val["classe"] != val["previsao"]]
pares = erros.groupby(["classe", "previsao"]).size().sort_values(ascending=False)
print()
print(f"Erros na validação: {milhar(len(erros))} de {milhar(len(val))} ({br(len(erros) / len(val) * 100, 2)}%)")
print("Pares (classe real, classe prevista) com erro:")
print(pares.to_string())
print()
print("Um erro concreto de cada par (colunas mediana, avg e linha_rotulo são só auditoria):")
for (real, prev), _ in pares.items():
    ex = erros[(erros["classe"] == real) & (erros["previsao"] == prev)].sort_values(["fluxo_id", "timestamp"]).iloc[0]
    print(f"\nreal {real} -> prevista {prev}")
    print(ex[cols_erro].to_string())
print()
print("Fluxos com mais erros:", erros["fluxo_id"].value_counts().head(5).to_dict())
print("Erros em fluxos de caminho curto (mediana < 5 ms):", int((erros["mediana"] < 5).sum()),
      "| de caminho longo (mediana > 150 ms):", int((erros["mediana"] > 150).sum()))
print("Linha da tabela da Tarefa 2 que definiu a classe real nos erros:")
print(pd.crosstab([erros["classe"], erros["previsao"]], erros["linha_rotulo"]).to_string())

(a) Caminho longo estável (mediana > 150 ms, |z_robusto| < 2, sem perda) previsto como FALHA: 0
(b) FALHA de caminho curto (mediana < 5 ms) prevista como OK: 0

(a): o caso não apareceu na validação.

(b): o caso não apareceu na validação.

Erros na validação: 113 de 38.138 (0,30%)
Pares (classe real, classe prevista) com erro:
classe  previsao
FALHA   OK          82
RISCO   OK          17
FALHA   RISCO       14

Um erro concreto de cada par (colunas mediana, avg e linha_rotulo são só auditoria):

real FALHA -> prevista OK
fluxo_id                 6659|103.179.107.33
datetime_utc       2026-09-17T02:20:07+00:00
mediana                           236.597873
avg                               237.092921
z_robusto                           0.266597
aumento_pct                         0.209236
jitter_relativo                     0.045049
perda_pct                          33.333333
n5_timeout                               0.0
n5_aumento80                             0.0
crit_risco_ind       

## 4. Florestas de contraste (nao sao o modelo do projeto)

Treinamos florestas no mesmo conjunto de treino, mantendo os hiperparametros selecionados, mas acrescentando o RTT absoluto (avg) ou a rota. Elas servem para medir se o modelo usaria esses atalhos quando disponiveis. Uma terceira variante usa somente RTT absoluto e rota, para medir o quanto a distancia sozinha explica a classe. As importancias sao agregadas sobre todas as arvores; raizes e arvores componentes exibidas sao apenas exemplos, nao regras completas da floresta.

- [ ] Comparar a importancia somada de RTT absoluto/rota e o F1 macro de validacao
- [ ] Declarar que as florestas de contraste ficam fora da entrega final

In [30]:
rotas = sorted(tr["rota"].unique())


def com_rota(d):
    dum = pd.get_dummies(d["rota"]).reindex(columns=rotas, fill_value=0).astype(int)
    dum.columns = [f"rota_{c}" for c in dum.columns]
    return dum


def montar(d, usar_relativas, usar_avg, usar_rota):
    partes = []
    if usar_relativas:
        partes.append(d[FEATURES])
    if usar_avg:
        partes.append(d[["avg"]])
    if usar_rota:
        partes.append(com_rota(d))
    return pd.concat(partes, axis=1)


variantes = [
    ("C1: metricas relativas + RTT absoluto (avg)", True, True, False),
    ("C2: metricas relativas + rota", True, False, True),
    ("C3 (extra): so RTT absoluto + rota", False, True, True),
]

for nome, rel, av, ro in variantes:
    Xt, Xv = montar(tr, rel, av, ro), montar(va, rel, av, ro)
    cont = nova_floresta(MAX_DEPTH, MIN_FOLHA).fit(Xt, y_tr)
    imp = pd.Series(cont.feature_importances_, index=Xt.columns)
    colunas_atalho = [c for c in Xt.columns if c == "avg" or c.startswith("rota_")]
    atalho = imp[colunas_atalho].sum()
    f1c = f1_score(y_va, cont.predict(Xv), labels=CLASSES, average="macro", zero_division=0)
    raizes_cont = pd.Series([
        Xt.columns[arvore.tree_.feature[0]]
        for arvore in cont.estimators_
        if arvore.tree_.feature[0] >= 0
    ]).value_counts()
    print("=" * 78)
    print(nome)
    print("Feature mais frequente na raiz:", raizes_cont.index[0], f"({raizes_cont.iloc[0]} de {len(cont.estimators_)} arvores)")
    print("Importancia somada de RTT absoluto/rota:", f"{br(atalho * 100, 1)}%",
          "| F1 macro na validacao (so contexto):", br(f1c, 4))
    print("Primeiras divisoes da primeira arvore componente (apenas exemplo):")
    print(texto_arvore_com_rotulos(cont, cont.estimators_[0], list(Xt.columns), max_depth=2))

print("Importancia somada de RTT absoluto/rota na floresta oficial:",
      f"{br(pd.Series(floresta.feature_importances_, index=FEATURES).reindex(['avg'], fill_value=0).sum() * 100, 1)}%")
print("F1 macro da floresta oficial (validacao):", br(f1_macro, 4))
print("Estas florestas de contraste NAO fazem parte da entrega final.")

C1: metricas relativas + RTT absoluto (avg)
Feature mais frequente na raiz: aumento_pct (37 de 100 arvores)
Importancia somada de RTT absoluto/rota: 6,8% | F1 macro na validacao (so contexto): 0,9956
Primeiras divisoes da primeira arvore componente (apenas exemplo):
|--- z_robusto <= 3.50
|   |--- aumento_pct <= 0.26
|   |   |--- perda_pct <= 16.67
|   |   |   |--- truncated branch of depth 4
|   |   |--- perda_pct >  16.67
|   |   |   |--- class: FALHA
|   |--- aumento_pct >  0.26
|   |   |--- n5_risco <= 1.50
|   |   |   |--- truncated branch of depth 4
|   |   |--- n5_risco >  1.50
|   |   |   |--- truncated branch of depth 4
|--- z_robusto >  3.50
|   |--- class: FALHA

C2: metricas relativas + rota
Feature mais frequente na raiz: z_robusto (22 de 100 arvores)
Importancia somada de RTT absoluto/rota: 7,2% | F1 macro na validacao (so contexto): 0,9921
Primeiras divisoes da primeira arvore componente (apenas exemplo):
|--- z_robusto <= 3.50
|   |--- n5_aumento80 <= 1.50
|   |   |--- 

## 5. Salvar o artefato da floresta oficial

In [31]:
PASTA_MODELOS.mkdir(parents=True, exist_ok=True)
params = {
    "modelo": "RandomForestClassifier",
    "criterio": CRITERIO,
    "n_estimators": N_ESTIMATORS,
    "max_depth": MAX_DEPTH,
    "min_samples_leaf": MIN_FOLHA,
    "random_state": SEMENTE,
    "n_jobs": -1,
    "profundidade_media_arvores": round(float(np.mean(profundidades)), 4),
    "folhas_medias_arvores": round(float(np.mean(folhas)), 4),
    "features": FEATURES,
    "alvo": ALVO,
    "treinada_em": "treino.csv",
    "escolha_de_hiperparametros": "validacao.csv (F1 macro, tolerancia %.3f)" % TOLERANCIA_F1,
    "f1_macro_validacao": round(float(f1_macro), 6),
    "scikit_learn": sklearn.__version__,
}
(PASTA_MODELOS / "floresta_oficial_params.json").write_text(
    json.dumps(params, ensure_ascii=False, indent=2), encoding="utf-8"
)
(PASTA_MODELOS / "floresta_oficial_arvore_exemplo.txt").write_text(
    texto_arvore, encoding="utf-8"
)
joblib.dump(floresta, PASTA_MODELOS / "floresta_oficial.joblib")
print("Salvo em", PASTA_MODELOS, ":",
      "floresta_oficial.joblib, floresta_oficial_params.json, floresta_oficial_arvore_exemplo.txt")

Salvo em C:\Users\lau-s\OneDrive\Documents\Modelo-preditivo---Projeto-Acad-mico---4-Semestre\notebook\data_ripe_atlas\models : floresta_oficial.joblib, floresta_oficial_params.json, floresta_oficial_arvore_exemplo.txt


## Checklist (marque no diario depois de conferir as saidas)

- [ ] Treino, validacao e teste sao os da Tarefa 2 (secao 1)
- [ ] A ficha de baseline nao foi recalculada com o Periodo B (secao 1)
- [ ] Lista de colunas e alvo conferidos (secao 1)
- [ ] Criterio, numero de arvores, max_depth, min_samples_leaf e semente declarados; escolha pela validacao (secao 2)
- [ ] Profundidade e folhas medias das arvores componentes, importancia das features e exemplo ilustrativo (secao 2)
- [ ] Matriz 3x3, precisao/recall/F1 por classe, F1 macro, dois erros concretos (secao 3)
- [ ] Florestas de contraste: importancia dos atalhos e declaracao de que ficam fora da entrega (secao 4)
- [ ] Board atualizado e tabela de integrantes (secao 5 do diario)